In [1]:
# add the parent directory to the system path
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.analysis.kp_alpha import KrippendorffSpanMatcher

In [3]:
annotator_paths = {
        "Caspar": r"C:\Users\norouzin\Desktop\JointLearning\datasets\annotators_agreement_dataset\caspar.jsonl",
        "Rasoul": r"C:\Users\norouzin\Desktop\JointLearning\datasets\annotators_agreement_dataset\rasoul.jsonl", 
        "Bennett": r"C:\Users\norouzin\Desktop\JointLearning\datasets\annotators_agreement_dataset\bennett.jsonl"
    }
    
print(f"\n📊 CONFIGURATION:")
print(f"• Annotators: {', '.join(annotator_paths.keys())}")
print(f"• Target labels: cause, effect")
print(f"• Matching method: Character offset partial overlap")
print(f"• Missing data handling: Excluded from analysis")


📊 CONFIGURATION:
• Annotators: Caspar, Rasoul, Bennett
• Target labels: cause, effect
• Matching method: Character offset partial overlap
• Missing data handling: Excluded from analysis


In [5]:
# Initialize matcher
matcher = KrippendorffSpanMatcher(annotator_paths)
print(f"• Total sentences: {matcher.num_sentences}")
    
# Verify label consistency
unique_labels = matcher.get_unique_labels()
print(f"\n📋 LABEL VERIFICATION:")
for annotator, labels in unique_labels.items():
    print(f"  {annotator}: {', '.join(sorted(labels))}")

print("\n" + "=" * 80)
print("AGREEMENT ANALYSIS RESULTS")
print("=" * 80)

# Run all analyses
results = matcher.run_all()

print("\n" + "=" * 80)
print("SUMMARY AND INTERPRETATION")
print("=" * 80)

# Summary statistics
agreement_stats = results["agreement_statistics"]
print(f"\n📈 OVERALL AGREEMENT SUMMARY:")
print(f"  Task 1 (Sentence Classification): α = {results['sentence_causality_alpha']:.4f}")
print(f"    • Overall agreement rate: {agreement_stats['overall_agreement_rate']:.1f}%")
print(f"    • Causal agreements: {agreement_stats['causal_agreements']} sentences")
print(f"    • Non-causal agreements: {agreement_stats['non_causal_agreements']} sentences")
print(f"    • Mixed disagreements: {agreement_stats['mixed_disagreements']} sentences")
    
print(f"\n  Task 2 (Span Agreement):")
print(f"    • Cause spans: α = {results['cause_span_alpha']:.4f}")
print(f"    • Effect spans: α = {results['effect_span_alpha']:.4f}")
    
print(f"\n  Task 3 (Link Agreement):")
print(f"    • 3A - Full structure: α = {results['link_alpha_full_structure']:.4f}")
print(f"    • 3B - Filtered mapping: α = {results['link_alpha_filtered_mapping']:.4f}")
    
# Key insights
print(f"\n🔍 KEY INSIGHTS:")
full_alpha = results['link_alpha_full_structure']
filtered_alpha = results['link_alpha_filtered_mapping']

print(f"  • Span vs Link Agreement:")
avg_span_alpha = (results['cause_span_alpha'] + results['effect_span_alpha']) / 2
print(f"    - Average span agreement: {avg_span_alpha:.4f}")
print(f"    - Full link agreement: {full_alpha:.4f}")
if avg_span_alpha > full_alpha:
    print(f"    → Links are harder to agree on than individual spans")

print(f"  • Error Attribution:")
if filtered_alpha > full_alpha:
    print(f"    - Filtered α ({filtered_alpha:.4f}) > Full α ({full_alpha:.4f})")
    print(f"    → Primary disagreement source: SPAN IDENTIFICATION")
    print(f"    → Secondary disagreement: linking strategy")
else:
    print(f"    - Full α ≥ Filtered α")
    print(f"    → Primary disagreement source: LINKING STRATEGY")

print(f"\n  • Best Pairwise Agreement:")
best_sentence_pair = max(results['pairwise_sentence'].items(), key=lambda x: x[1]['alpha'])
best_cause_pair = max(results['pairwise_cause'].items(), key=lambda x: x[1]['alpha'])
best_effect_pair = max(results['pairwise_effect'].items(), key=lambda x: x[1]['alpha'])

print(f"    - Sentence level: {best_sentence_pair[0]} (α={best_sentence_pair[1]['alpha']:.4f})")
print(f"    - Cause spans: {best_cause_pair[0]} (α={best_cause_pair[1]['alpha']:.4f})")
print(f"    - Effect spans: {best_effect_pair[0]} (α={best_effect_pair[1]['alpha']:.4f})")

# Methodological notes
print(f"\n📋 METHODOLOGICAL NOTES:")
print(f"  • Character offset partial overlap: Any character overlap = perfect match")
print(f"  • Sentence classification: Binary (causal=1, non-causal=0)")
print(f"  • Span agreement: Filtered to sentences all annotators marked causal")
print(f"  • Link agreement: Cartesian product of cause×effect spans per sentence")
print(f"  • Task 3A: Structural agreement including span and link disagreements")
print(f"  • Task 3B: Link mapping agreement over shared spans only")

print(f"\n✅ ANALYSIS COMPLETE")
print("=" * 80)

• Total sentences: 60

📋 LABEL VERIFICATION:
  Caspar: cause, effect
  Rasoul: cause, effect
  Bennett: cause, effect

AGREEMENT ANALYSIS RESULTS
--- Task 1: Overall Sentence Agreement ---
Krippendorff's Alpha (Overall Agreement): 0.6490
Overall Agreement Rate: 76.7% (46/60 sentences)
  • All annotators agreed 'causal': 33 sentences
  • All annotators agreed 'non-causal': 13 sentences
  • Mixed disagreements: 14 sentences
Note: Both causal and non-causal agreements contribute positively to overall agreement.
Sentences where all annotators agreed on 'causal' (for span/link analysis): 33

--- Task 2: Cause and Effect Span Agreement (Character Offset Partial Overlap) ---
Krippendorff's Alpha (Cause Spans, filtered): 0.8294
Krippendorff's Alpha (Effect Spans, filtered): 0.9153

--- Task 3: Cause-Effect Link Agreement (Character Offset Partial Overlap) ---
Task 3A - Full Structure Agreement: 0.6683
  (Do annotators agree on the full causal structure including spans and links?)
Task 3B - Fil